## UrbanRide_07 — Gold Customer Features
**Method:** Batch aggregation — Silver → Gold  
**Inputs:**
- `urbanride.silver.customers` — demographics, churn, rating
- `urbanride.silver.trips` — trip behaviour aggregations
- `urbanride.silver.payments` — spend behaviour aggregations

**Output:** `urbanride.gold.customer_features`  
**Schedule:** Daily · runs after UrbanRide_06 (Silver layer must be complete)  
**Partition:** `city`  
**ZORDER:** `customer_id`, `is_churned`

**Grain:** one row per `customer_id`

**Feature engineering:**
- Direct attributes from silver.customers (city, device_type, referral, churn)
- Derived: `is_referred`, `customer_age_days`, `days_since_last_trip`
- Trip aggregations: total/completed/cancelled trips, recency, distance, duration, surge, vehicle preference, day/weather patterns
- Payment aggregations: total spend, avg spend, preferred method, failed payments

**Feeds:** Churn Prediction Model (11) · Customer Segmentation Model (14)

In [0]:
from pyspark.sql.functions import (
    col, when, lit, current_timestamp, current_date,
    count, sum, avg, min, max, round, datediff,
    broadcast, coalesce, expr
)

CATALOG = 'urbanride'
SILVER  = f'{CATALOG}.silver'
GOLD    = f'{CATALOG}.gold'

spark.sql(f'USE CATALOG {CATALOG}')

print(f'Catalog : {CATALOG}')
print(f'Sources : {SILVER}.customers, {SILVER}.trips, {SILVER}.payments')
print(f'Target  : {GOLD}.customer_features')
print('Config ready.')


Catalog : urbanride
Sources : urbanride.silver.customers, urbanride.silver.trips, urbanride.silver.payments
Target  : urbanride.gold.customer_features
Config ready.


In [0]:
# Gold customer_features is a full recompute on each run
# Reason: aggregations like days_since_last_trip, customer_age_days change daily
# even if no new Silver rows were added — a stale Gold row would give wrong churn signals
# Strategy: overwrite full table daily — 89K rows is small, cost is negligible

print('Checking load type...')

table_exists = spark.catalog.tableExists(f'{GOLD}.customer_features')

if not table_exists:
    LOAD_TYPE = 'full'
    print('  Gold table does not exist — FULL LOAD')
else:
    LOAD_TYPE = 'full'
    last_run = spark.sql(
        f'SELECT MAX(_processed_at) FROM {GOLD}.customer_features'
    ).collect()[0][0]
    print(f'  Gold table exists — FULL RECOMPUTE (overwrite)')
    print(f'  Last processed at : {last_run}')

print(f'  Load type : {LOAD_TYPE}')
print()
print('  NOTE: Gold customer_features always does a full recompute.')
print('  Reason: days_since_last_trip and customer_age_days change daily')
print('  even without new Silver data — incremental would produce stale features.')


Checking load type...
  Gold table does not exist — FULL LOAD
  Load type : full

  NOTE: Gold customer_features always does a full recompute.
  Reason: days_since_last_trip and customer_age_days change daily
  even without new Silver data — incremental would produce stale features.


In [0]:
print('Reading Silver tables...')

df_customers = spark.table(f'{SILVER}.customers')
df_trips     = spark.table(f'{SILVER}.trips')
df_payments  = spark.table(f'{SILVER}.payments')

print(f'  silver.customers : {df_customers.count():,} rows')
print(f'  silver.trips     : {df_trips.count():,} rows')
print(f'  silver.payments  : {df_payments.count():,} rows')

# Filter trips — only use valid completed trips for behaviour aggregations
# Exclude ghost trips, invalid distance, invalid duration
df_trips_clean = df_trips.filter(
    (col('trip_status') == 'completed') &
    (col('is_ghost_trip')       == False) &
    (col('is_distance_invalid') == False) &
    (col('is_duration_invalid') == False)
)

# Filter payments — only successful payments for spend aggregations
df_payments_clean = df_payments.filter(
    (col('payment_status') == 'success') &
    (col('is_orphan_payment') == False)
)

print()
print(f'  trips (clean completed) : {df_trips_clean.count():,} rows')
print(f'  payments (clean success): {df_payments_clean.count():,} rows')


Reading Silver tables...
  silver.customers : 89,041 rows
  silver.trips     : 21,359,745 rows
  silver.payments  : 19,600,753 rows

  trips (clean completed) : 19,444,155 rows
  payments (clean success): 18,956,746 rows


In [0]:
print('Building customer base features...')

df_base = df_customers.select(
    col('customer_id'),
    col('city'),
    col('device_type'),
    col('referral_source'),
    col('signup_date'),
    col('rating').alias('customer_rating'),
    col('is_churned'),
    col('churn_date'),
    col('is_rating_invalid'),
    col('is_churn_inconsistent'),

    # Derived: is_referred — simpler boolean for ML than raw UUID
    when(col('referred_by_customer_id').isNotNull(), True)
        .otherwise(False)
        .alias('is_referred'),

    # Derived: customer_age_days — how long they have been a customer
    # Key churn signal — newer customers churn more
    datediff(current_date(), col('signup_date')).alias('customer_age_days'),
)

print(f'  Base feature rows : {df_base.count():,}')
print(f'  Columns           : {len(df_base.columns)}')


Building customer base features...
  Base feature rows : 89,041
  Columns           : 12


In [0]:
print('Building trip behaviour aggregations...')

# ── Total and completed trip counts ───────────────────────────────────────────
df_trip_counts = df_trips.groupBy('customer_id').agg(
    count('trip_id').alias('total_trips'),
    count(when(col('trip_status') == 'completed', 1)).alias('completed_trips'),
    count(when(col('trip_status') == 'cancelled', 1)).alias('cancelled_trips'),
)

# Cancellation rate — high rate is a churn signal
df_trip_counts = df_trip_counts.withColumn(
    'cancellation_rate',
    round(col('cancelled_trips') / col('total_trips'), 4)
)

# ── Recency and behaviour from clean completed trips only ─────────────────────
df_trip_agg = df_trips_clean.groupBy('customer_id').agg(
    min('pickup_time').cast('date').alias('first_trip_date'),
    max('pickup_time').cast('date').alias('last_trip_date'),
    round(avg('distance_km'), 2).alias('avg_trip_distance_km'),
    round(avg('trip_duration_minutes'), 2).alias('avg_trip_duration_minutes'),
    round(avg('surge_multiplier'), 4).alias('avg_surge_multiplier'),

    # Weekend ratio — do they ride more on weekends?
    round(
        count(when(col('day_type') == 'weekend', 1)) / count('trip_id'), 4
    ).alias('weekend_trip_ratio'),

    # Rainy day ratio — are they weather dependent?
    round(
        count(when(col('weather') == 'Rain', 1)) / count('trip_id'), 4
    ).alias('rainy_day_trip_ratio'),
)

# Derived: days_since_last_trip — strongest single churn signal
df_trip_agg = df_trip_agg.withColumn(
    'days_since_last_trip',
    datediff(current_date(), col('last_trip_date'))
)

# ── Favourite vehicle type — most used vehicle type per customer ───────────────
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df_vehicle = df_trips_clean.groupBy('customer_id', 'vehicle_type') \
    .agg(count('trip_id').alias('vtype_count'))

window_vtype = Window.partitionBy('customer_id').orderBy(col('vtype_count').desc())

df_favourite_vehicle = df_vehicle \
    .withColumn('rn', row_number().over(window_vtype)) \
    .filter(col('rn') == 1) \
    .select('customer_id', col('vehicle_type').alias('favourite_vehicle_type'))

print(f'  Trip counts agg    : {df_trip_counts.count():,} customers')
print(f'  Trip behaviour agg : {df_trip_agg.count():,} customers')
print(f'  Favourite vehicle  : {df_favourite_vehicle.count():,} customers')


Building trip behaviour aggregations...
  Trip counts agg    : 89,041 customers
  Trip behaviour agg : 89,041 customers
  Favourite vehicle  : 89,041 customers


In [0]:
print('Building payment behaviour aggregations...')

# ── Spend aggregations from clean successful payments ─────────────────────────
df_payment_agg = df_payments_clean.groupBy('customer_id').agg(
    round(sum('amount'), 2).alias('total_spend'),
    round(avg('amount'), 2).alias('avg_spend_per_trip'),
)

# ── Failed payment stats from all payments ────────────────────────────────────
df_payment_counts = df_payments.groupBy('customer_id').agg(
    count('payment_id').alias('total_payments'),
    count(when(col('payment_status') == 'failed', 1)).alias('failed_payment_count'),
)

df_payment_counts = df_payment_counts.withColumn(
    'failed_payment_rate',
    round(col('failed_payment_count') / col('total_payments'), 4)
)

# ── Preferred payment method — most used method per customer ──────────────────
df_pay_method = df_payments_clean.groupBy('customer_id', 'payment_method') \
    .agg(count('payment_id').alias('method_count'))

window_method = Window.partitionBy('customer_id').orderBy(col('method_count').desc())

df_preferred_method = df_pay_method \
    .withColumn('rn', row_number().over(window_method)) \
    .filter(col('rn') == 1) \
    .select('customer_id', col('payment_method').alias('preferred_payment_method'))

print(f'  Payment spend agg    : {df_payment_agg.count():,} customers')
print(f'  Payment counts agg   : {df_payment_counts.count():,} customers')
print(f'  Preferred method     : {df_preferred_method.count():,} customers')


Building payment behaviour aggregations...
  Payment spend agg    : 89,041 customers
  Payment counts agg   : 89,041 customers
  Preferred method     : 89,041 customers


In [0]:
print('Joining all feature sets...')

# All aggregated tables are at customer_id grain (~89K rows)
# All joins are broadcast-eligible after aggregation
# Base (customers) is the spine — left join everything to preserve all customers
# even those with no trips yet (new customers)

df_gold = df_base \
    .join(broadcast(df_trip_counts),      on='customer_id', how='left') \
    .join(broadcast(df_trip_agg),         on='customer_id', how='left') \
    .join(broadcast(df_favourite_vehicle),on='customer_id', how='left') \
    .join(broadcast(df_payment_agg),      on='customer_id', how='left') \
    .join(broadcast(df_payment_counts),   on='customer_id', how='left') \
    .join(broadcast(df_preferred_method), on='customer_id', how='left')

# Fill nulls for customers with no trips or payments
# (new customers who signed up but never rode)
df_gold = df_gold \
    .fillna(0, subset=[
        'total_trips', 'completed_trips', 'cancelled_trips',
        'cancellation_rate', 'total_spend', 'avg_spend_per_trip',
        'total_payments', 'failed_payment_count', 'failed_payment_rate',
        'avg_trip_distance_km', 'avg_trip_duration_minutes',
        'avg_surge_multiplier', 'weekend_trip_ratio',
        'rainy_day_trip_ratio', 'days_since_last_trip'
    ]) \
    .fillna('Unknown', subset=['favourite_vehicle_type', 'preferred_payment_method'])

# Add metadata
df_gold = df_gold \
    .withColumn('_processed_at', current_timestamp()) \
    .withColumn('_source', lit('silver.customers+trips+payments'))

# Drop internal payment count column not needed in Gold
df_gold = df_gold.drop('total_payments')

print(f'  Final row count   : {df_gold.count():,}')
print(f'  Final column count: {len(df_gold.columns)}')
print()
print('Gold schema:')
df_gold.printSchema()


Joining all feature sets...
  Final row count   : 89,041
  Final column count: 32

Gold schema:
root
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- referral_source: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- is_churned: boolean (nullable = true)
 |-- churn_date: date (nullable = true)
 |-- is_rating_invalid: boolean (nullable = true)
 |-- is_churn_inconsistent: boolean (nullable = true)
 |-- is_referred: boolean (nullable = false)
 |-- customer_age_days: integer (nullable = true)
 |-- total_trips: long (nullable = false)
 |-- completed_trips: long (nullable = false)
 |-- cancelled_trips: long (nullable = false)
 |-- cancellation_rate: double (nullable = false)
 |-- first_trip_date: date (nullable = true)
 |-- last_trip_date: date (nullable = true)
 |-- avg_trip_distance_km: double (nullable = false)
 |-- avg_trip_duration_minutes: do

In [0]:
import time
print(f'Writing gold.customer_features — mode: overwrite...')
t0 = time.time()

df_gold.write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('city') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD}.customer_features')

print(f'  Rows written : {df_gold.count():,}')
print(f'  Write time   : {time.time()-t0}s')
print()
print('Running OPTIMIZE + ZORDER...')
spark.sql(f'OPTIMIZE {GOLD}.customer_features ZORDER BY (customer_id, is_churned)')
print('  OPTIMIZE + ZORDER complete')


Writing gold.customer_features — mode: overwrite...
  Rows written : 89,041
  Write time   : 18.14443826675415s

Running OPTIMIZE + ZORDER...
  OPTIMIZE + ZORDER complete


In [0]:
print('=== GOLD CUSTOMER FEATURES VERIFICATION ===')
print()

df_verify = spark.table(f'{GOLD}.customer_features')
total = df_verify.count()

print(f'  Total rows    : {total:,}')
print(f'  Total columns : {len(df_verify.columns)}')
print()

# Data quality checks
print('Data quality checks:')
checks = [
    ('Null customer_id',         df_verify.filter(col('customer_id').isNull()).count()),
    ('Null city',                df_verify.filter(col('city').isNull()).count()),
    ('Null is_churned',          df_verify.filter(col('is_churned').isNull()).count()),
    ('Null total_trips',         df_verify.filter(col('total_trips').isNull()).count()),
    ('Null days_since_last_trip',df_verify.filter(col('days_since_last_trip').isNull()).count()),
    ('Null total_spend',         df_verify.filter(col('total_spend').isNull()).count()),
    ('Negative total_spend',     df_verify.filter(col('total_spend') < 0).count()),
    ('Cancellation rate > 1',    df_verify.filter(col('cancellation_rate') > 1).count()),
]

all_passed = True
for check, result in checks:
    status = 'PASS' if result == 0 else 'WARN'
    if result > 0: all_passed = False
    print(f'  {status}  {check:<35} : {result:,}')

print()
print('Churn distribution:')
df_verify.groupBy('is_churned').count().orderBy('is_churned').show()

print('City distribution:')
df_verify.groupBy('city').count().orderBy('count', ascending=False).show()

print('Device type distribution:')
df_verify.groupBy('device_type').count().orderBy('count', ascending=False).show()

print('Favourite vehicle type distribution:')
df_verify.groupBy('favourite_vehicle_type').count().orderBy('count', ascending=False).show()

print('Preferred payment method distribution:')
df_verify.groupBy('preferred_payment_method').count().orderBy('count', ascending=False).show()

print('Key feature stats:')
df_verify.select(
    'total_trips', 'completed_trips', 'cancelled_trips',
    'cancellation_rate', 'days_since_last_trip',
    'total_spend', 'avg_spend_per_trip', 'avg_surge_multiplier'
).summary('min', 'max', 'mean', '50%').show(truncate=False)

print('Churned vs non-churned — avg feature comparison:')
df_verify.groupBy('is_churned').agg(
    round(avg('total_trips'), 1).alias('avg_total_trips'),
    round(avg('days_since_last_trip'), 1).alias('avg_days_since_last_trip'),
    round(avg('total_spend'), 2).alias('avg_total_spend'),
    round(avg('cancellation_rate'), 4).alias('avg_cancellation_rate'),
    round(avg('avg_surge_multiplier'), 4).alias('avg_surge_multiplier'),
).orderBy('is_churned').show()

print('Sample rows (churned customers):')
df_verify.filter(col('is_churned') == True).select(
    'customer_id', 'city', 'is_churned', 'total_trips',
    'days_since_last_trip', 'total_spend', 'cancellation_rate',
    'favourite_vehicle_type', 'preferred_payment_method'
).limit(5).show(truncate=False)

print()
detail = spark.sql(f'DESCRIBE DETAIL {GOLD}.customer_features').collect()[0]
print(f'  numFiles    : {detail["numFiles"]}')
print(f'  sizeInBytes : {detail["sizeInBytes"]:,}')

print()
if all_passed:
    print('All quality checks passed. Gold customer features ready.')
else:
    print('Some checks flagged — review WARN items above.')
print('Next: UrbanRide_08 — Gold Trip Features')


=== GOLD CUSTOMER FEATURES VERIFICATION ===

  Total rows    : 89,041
  Total columns : 32

Data quality checks:
  PASS  Null customer_id                    : 0
  PASS  Null city                           : 0
  PASS  Null is_churned                     : 0
  PASS  Null total_trips                    : 0
  PASS  Null days_since_last_trip           : 0
  PASS  Null total_spend                    : 0
  PASS  Negative total_spend                : 0
  PASS  Cancellation rate > 1               : 0

Churn distribution:
+----------+-----+
|is_churned|count|
+----------+-----+
|     false|78271|
|      true|10770|
+----------+-----+

City distribution:
+---------+-----+
|     city|count|
+---------+-----+
|    Delhi|26619|
|   Mumbai|22313|
|Bengaluru|17797|
|Hyderabad|13434|
|     Pune| 8878|
+---------+-----+

Device type distribution:
+-----------+-----+
|device_type|count|
+-----------+-----+
|    Android|62014|
|        iOS|22507|
|        Web| 4520|
+-----------+-----+

Favourite vehicle 

### Two Issues to Investigate
###  Issue 1 — Favourite vehicle type:   
Sedan | 70,311   
Auto  | 18,730   
SUV   | 0   

**`SUV is completely missing.`** Every customer's favourite is either Sedan or Auto. **SUV exists in Silver trips (15% of trips) but no customer took enough SUVs to make it their favourite**. Not a bug — just reflects behaviour. But worth noting for the segmentation model.


### Issue 2 — Preferred payment method:
UPI | 89,041    
**Every single customer's preferred method is UPI** — no Credit Card, Wallet or Cash anywhere.

### Churned vs Non-Churned Comparison — This is the most valuable output
| Feature | Active | Churned | Signal? |
|---|---|---|---|
| avg_total_trips | 250.6 | 161.7 | ✅ Strong — churned took 35% fewer trips |
| avg_days_since_last_trip | 9.2 | 71.7 | ✅ Very strong — churned haven't ridden in 72 days vs 9 |
| avg_total_spend | 31,974 | 20,692 | ✅ Strong — churned spent 35% less |
| avg_cancellation_rate | 0.085 | 0.086 | ❌ Weak — almost identical, not a churn signal |
| avg_surge_multiplier | 1.3126 | 1.3218 | ❌ Weak — nearly identical |